# Chapitre 13 - Méthodes à prototypes (cours 1)

Ce premier notebook du chapitre 13 présente les **méthodes à prototypes** pour la
classification :

1. le **K-means** utilisé comme classifieur (section 13.2.1 du livre) ;
2. la **quantification vectorielle par apprentissage** (LVQ, section 13.2.2) ;
3. les **mélanges gaussiens** vus comme des prototypes « souples » (section 13.2.3) ;
4. l'étude comparative « facile / difficile » de la section 13.3.1 (partie K-means et LVQ).

Nous reproduisons les figures 13.1, 13.2 et le panneau de droite de la figure 13.5.
Les plus proches voisins font l'objet du second notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl.plotting import decision_grid
from esl import datasets
from esl import ch13_prototypes as cp
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
set_style()

## 1. Cadre et notations

On dispose de $N$ couples d'apprentissage $(x_1, g_1), \dots, (x_N, g_N)$ où
$x_i \in \mathbb{R}^p$ est un vecteur de caractéristiques et $g_i \in \{1, \dots, K\}$
une étiquette de classe.

Une **méthode à prototypes** résume les données par un petit nombre de points
$m_1, \dots, m_M \in \mathbb{R}^p$, les *prototypes*, chacun muni d'une classe.
Un nouveau point $x$ est affecté à la classe du **prototype le plus proche** :

$$\hat G(x) = \text{classe de } m_{j^*}, \qquad j^* = \arg\min_{j} \|x - m_j\|.$$

La distance est la distance euclidienne, calculée après avoir **standardisé**
chaque variable (moyenne nulle, variance 1 sur l'échantillon d'apprentissage),
pour que des variables mesurées dans des unités différentes pèsent autant.

> **Intuition.** La règle « prototype le plus proche » découpe l'espace en
> **cellules de Voronoï** : la cellule de $m_j$ est l'ensemble des points plus
> proches de $m_j$ que de tous les autres prototypes. Chaque cellule est un polyèdre
> convexe (intersection de demi-espaces $\{x : \|x-m_j\| \le \|x - m_l\|\}$,
> et cette inégalité est linéaire en $x$ après développement des carrés).
> La frontière de décision est donc **linéaire par morceaux**.

Toute la difficulté est de choisir **combien** de prototypes utiliser et **où** les placer.

## 2. Une simulation à trois classes

Pour reproduire l'esprit de la figure 13.1, nous simulons trois classes en
dimension 2, chacune étant un **mélange de gaussiennes** : la classe $k$ possède
10 centres $\mu_{k1}, \dots, \mu_{k,10}$ (tirés autour d'un point de base) ;
une observation de la classe $k$ choisit un centre au hasard puis ajoute un
bruit $\mathcal N(0, \sigma^2 I_2)$.

Comme nous connaissons le modèle, nous pouvons calculer la **règle de Bayes**
$\arg\max_k \Pr(G = k \mid X = x)$ et sa frontière (courbe en pointillés violets
dans les figures).

In [2]:
rng = np.random.RandomState(2)
base = np.array([[-1.5, 1.0], [1.5, 1.0], [0.0, -1.5]])
means3 = [b + 0.8 * rng.randn(10, 2) for b in base]
sigma2_3 = 0.2
X3, y3 = cp.sample_gaussian_mixture(means3, 100, sigma2_3, rng)
X3_test, y3_test = cp.sample_gaussian_mixture(means3, 3000, sigma2_3, rng)
bayes3 = np.mean(cp.mixture_posterior(X3_test, means3, sigma2_3).argmax(1) != y3_test)
print("Apprentissage :", X3.shape, " test :", X3_test.shape)
print("Erreur de Bayes (estimée sur l'échantillon test) : %.3f" % bayes3)

Apprentissage : (300, 2)  test : (9000, 2)
Erreur de Bayes (estimée sur l'échantillon test) : 0.073


Une petite fonction de tracé, réutilisée dans tout le notebook : elle colore les
régions de décision d'un classifieur sur une grille, superpose les données, les
prototypes (gros points cerclés de noir) et la frontière de Bayes.

In [3]:
CMAP3 = ListedColormap(COLORS[:3])
PURPLE = COLORS[6]


def plot_regions(ax, predict, X, y, protos=None, proto_labels=None, bayes=None,
                 xlim=None, ylim=None, n=250, ncls=3):
    xlim = xlim or (X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
    ylim = ylim or (X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
    xx, yy, grid = decision_grid(xlim, ylim, n)
    cmap = ListedColormap(COLORS[:ncls])
    Z = predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=np.arange(ncls + 1) - 0.5, cmap=cmap, alpha=0.15)
    ax.contour(xx, yy, Z, levels=np.arange(ncls) + 0.5, colors="k", linewidths=1)
    if bayes is not None:
        B = bayes(grid).reshape(xx.shape)
        ax.contour(xx, yy, B, levels=np.arange(ncls) + 0.5, colors=PURPLE,
                   linewidths=1.5, linestyles="--")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap, s=12, alpha=0.7, vmin=0, vmax=ncls - 1)
    if protos is not None:
        ax.scatter(protos[:, 0], protos[:, 1], c=proto_labels, cmap=cmap, s=90,
                   edgecolors="k", linewidths=1.5, vmin=0, vmax=ncls - 1)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_xlabel("$X_1$")
    ax.set_ylabel("$X_2$")


bayes3_rule = lambda G: cp.mixture_posterior(G, means3, sigma2_3).argmax(1)